In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)


In [ ]:
# Table 2: Trial outcomes in participants with diabetes, hypertension, or both

In [ ]:
import statsmodels.api as sm
from intecomm_analytics.dataframes import get_df_main_1858
from intecomm_analytics.constants import HIV_ALONE
from intecomm_rando.constants import COMMUNITY_ARM, FACILITY_ARM
from statsmodels.stats.proportion import proportion_confint, proportions_ztest



In [ ]:
df_main = get_df_main_1858(None)


In [ ]:
cohort_filter = (df_main.primary_cohort==HIV_ALONE)
controlled_col= 'vl_endline_suppressed'

In [ ]:
df = df_main[cohort_filter].copy()
# df = df_main.copy()
df["group"] = df["assignment"]
df['group'] =  pd.Categorical(df["group"], categories=[FACILITY_ARM, COMMUNITY_ARM], ordered=True)

events_a = df[df['group'] == COMMUNITY_ARM][controlled_col].sum()
events_b = df[df['group'] == FACILITY_ARM][controlled_col].sum()
# Calculate the number of events and total individuals in each group
total_a = df[df['group'] == COMMUNITY_ARM].shape[0]
total_b = df[df['group'] == FACILITY_ARM].shape[0]

# Calculate risk in each group
risk_a = events_a / total_a
risk_b = events_b / total_b

# Calculate crude risk difference
crude_risk_difference = risk_a - risk_b

print(f"The crude risk difference between Group A and Group B is {crude_risk_difference:.4f}")


In [ ]:
df = df_main[cohort_filter].copy()
# df = df_main.copy()
df["group"] = df["assignment"]
df['group'] =  pd.Categorical(df["group"], categories=[FACILITY_ARM, COMMUNITY_ARM], ordered=True)

# Create a contingency table
contingency_table = pd.crosstab(df['group'], df[controlled_col])
contingency_table

In [ ]:
df = df_main[cohort_filter].copy()
df["group"] = df["assignment"]
df['group'] =  pd.Categorical(df["group"], categories=[FACILITY_ARM, COMMUNITY_ARM], ordered=True)

events_a = df[df['group'] == COMMUNITY_ARM][controlled_col].sum()
events_b = df[df['group'] == FACILITY_ARM][controlled_col].sum()
# Calculate the number of events and total individuals in each group
total_a = df[df['group'] == COMMUNITY_ARM].shape[0]
total_b = df[df['group'] == FACILITY_ARM].shape[0]

# Calculate risk in each group
risk_a = events_a / total_a
risk_b = events_b / total_b

# Calculate crude risk difference
crude_risk_difference = risk_a - risk_b
crude_risk_difference

In [ ]:
contingency_table = pd.crosstab(df['group'], df[controlled_col])

# Calculate risk difference using statsmodels
sm.stats.Table2x2(contingency_table.values).summary()

# ci_low, ci_high = sm.stats.Table2x2(contingency_table.values).risk_difference_confint()
# p_value = sm.stats.Table2x2(contingency_table.values).risk_difference_pvalue()
#
# print(f"Crude Risk Difference: {risk_diff:.2f}")
# print(f"95% Confidence Interval: ({ci_low:.2f}, {ci_high:.2f})")
# print(f"p-value: {p_value:.4f}")

In [ ]:
# Perform z-test for proportions
count = [events_a, events_b]
nobs = [total_a, total_b]
stat, p_value = proportions_ztest(count, nobs)

# Calculate confidence intervals for each group
ci_A_low, ci_A_high = proportion_confint(events_a, total_a, alpha=0.05)
ci_B_low, ci_B_high = proportion_confint(events_b, total_b, alpha=0.05)

# Calculate confidence interval for the risk difference
ci_low = ci_A_low - ci_B_high
ci_high = ci_A_high - ci_B_low
print(f"Crude Risk Difference: {crude_risk_difference:.2f}")
print(f"95% Confidence Interval: ({ci_low:.2f}, {ci_high:.2f})")
print(f"p-value: {p_value:.4f}")

In [ ]:
sm.stats.proportion_confint(count[0], nobs[0], alpha=0.05)

In [ ]:
sm.stats.proportion_confint(count[1], nobs[1], alpha=0.05)

In [ ]:
df = df_main[df_main.primary_cohort==HIV_ALONE].copy()
df["group"] = df["assignment"]
df['group'] =  pd.Categorical(df["group"], categories=[FACILITY_ARM, COMMUNITY_ARM], ordered=True)
df["sex"] = df["gender"].apply(lambda x: 1 if x=="Female" else 0)
df = df.sort_values(by=["group"], ascending=True)
df = df.reset_index(drop=True)
X = df[['group', 'age_in_years', 'sex']]
X = pd.get_dummies(X, drop_first=True)  # Convert categorical variables to dummy variables
X = X.astype(float)
X
# df[['group', 'age_in_years', 'sex']]

In [ ]:
# viral load suppression
df[controlled_col] = df[controlled_col].fillna(0)
# df['vl_endline_notsuppressed'] = df['vl_endline_suppressed'].apply(lambda x: 1 if x == 0 else 0)
df = df.reset_index(drop=True)
y = df[controlled_col].astype(float)

In [ ]:
model1 = sm.GLM(y, sm.add_constant(X), family=sm.families.Binomial(link=sm.families.links.Logit())).fit()
print(model1.summary())

In [ ]:
coefficients = model1.params
conf_intervals = f"{model1.conf_int().loc["group_a"][0]:.2f} to {model1.conf_int().loc["group_a"][1]:.2f}"
pvalue =  model1.pvalues.loc["group_a"]

print(
    f"""
    The adjusted risk difference for control ({controlled_col}) between group A and group B was {coefficients['group_a']:.4f} (95% CI: {conf_intervals}, p = {pvalue:.4f}).\n
    This indicates that, after adjusting for age and sex, the risk of uncontrolled VL in group A is {coefficients['group_a'] * 100:.2f} percentage points lower\n
    than in group B. However, this difference is not statistically significant.\n""")

In [ ]:
model1.pvalues.loc["group_a"]

In [ ]:
# Extract adjusted risk difference
adjusted_risk_diff = model1.params
print(f"Adjusted Risk Difference: {adjusted_risk_diff}")

In [ ]:
# Extracting coefficients
coefficients = model1.params
print(f"Coefficients:\n{coefficients}")

# Extracting p-values
p_values = model1.pvalues
print(f"P-values:\n{p_values}")

# Extracting standard errors
standard_errors = model1.bse
print(f"Standard Errors:\n{standard_errors}")

# Extracting confidence intervals
conf_intervals = model1.conf_int()
print(f"Confidence Intervals:\n{conf_intervals}")

# Extracting the coefficient, p-value, and confidence intervals for group A
risk_diff = coefficients['group_a']
p_value = p_values['group_a']
conf_interval = conf_intervals.loc['group_a']

print(f"Estimate of Risk Difference: {risk_diff}")
print(f"95% Confidence Interval: {conf_interval}")
print(f"P-value: {p_value}")

In [ ]:
model2 = sm.GLM(y, sm.add_constant(X), family=sm.families.Binomial(link=sm.families.links.Identity())).fit()
print(model2.summary())


In [ ]:
# Extract adjusted risk difference
adjusted_risk_diff = model2.params
print(f"Adjusted Risk Difference: {adjusted_risk_diff}")

In [ ]:
y.dtypes

In [ ]:
y

In [ ]:
model.params

In [ ]:
model.pvalues


In [ ]:
model.bse

In [ ]:
model.tvalues

In [ ]:
model.conf_int()

In [ ]:
X.group_b.value_counts()

In [ ]:
X

In [ ]:
import statsmodels.stats.api as sms

# Parameters for the power analysis
effect_size = 0.1  # Example effect size (Cohen's d)
alpha = 0.05  # Significance level
power = 0.8  # Desired power

# Calculate the sample size
sample_size = sms.TTestIndPower().solve_power(effect_size=effect_size, alpha=alpha, power=power)
print(f"Required sample size: {sample_size:.2f}")



In [ ]:
icc = 0.02  # Example ICC value
adjusted_sample_size = sample_size / (1 + (sample_size - 1) * icc)
adjusted_sample_size

In [ ]:
(1 + ((sample_size - 1) * 0.02))

In [ ]:
# Given parameters from the paragraph
control_rate = 0.50  # 50% control rate in the control arm
intervention_rate = 0.60  # 60% control rate in the intervention arm (10% superiority)
alpha = 0.05  # Significance level (5%)
power = 0.80  # Desired power (80%)
icc = 0.02  # Intra-class correlation coefficient (ICC)
total_participants = 928  # Total participants
groups = 116  # Number of groups
participants_per_group = 8  # Participants per group

# Calculate the effect size for proportions
effect_size = sms.proportion_effectsize(control_rate, intervention_rate)

# Calculate the sample size without considering ICC
sample_size_without_icc = sms.NormalIndPower().solve_power(effect_size=effect_size, alpha=alpha, power=power)

# Adjust the sample size considering ICC
adjusted_sample_size = sample_size_without_icc / (1 + (sample_size_without_icc - 1) * icc)

# Print the calculated values
print(f"Effect Size: {effect_size:.4f}")
print(f"Sample Size without ICC: {sample_size_without_icc:.2f}")
print(f"Adjusted Sample Size per group: {adjusted_sample_size:.2f}")

# Verify the total number of participants
calculated_total_participants = groups * participants_per_group
print(f"Calculated Total Participants: {calculated_total_participants}")

# Check if the calculated total participants match the given total participants
if calculated_total_participants == total_participants:
    print("The calculated total participants match the given total participants.")
else:
    print("The calculated total participants do not match the given total participants.")


In [ ]:
import statsmodels.stats.api as sms

# Parameters for the power analysis for hypertension patients
control_rate_hypertension = 0.50  # 50% control rate in the control arm
intervention_rate_hypertension = 0.60  # 60% control rate in the intervention arm (10% superiority)
alpha = 0.05  # Significance level
power = 0.80  # Desired high power level
icc = 0.02  # ICC less than 0.02
loss_to_followup_rate = 0.10  # 5% loss to follow-up rate

# Calculate the effect size for proportions for hypertension patients
effect_size_hypertension = sms.proportion_effectsize(control_rate_hypertension, intervention_rate_hypertension)

# Calculate the sample size for hypertension patients
sample_size_hypertension = sms.NormalIndPower().solve_power(effect_size=effect_size_hypertension, alpha=alpha, power=power)
adjusted_sample_size_hypertension = sample_size_hypertension / (1 + (sample_size_hypertension - 1) * icc)

# Adjust the sample size for loss to follow-up
adjusted_sample_size_hypertension_with_loss = adjusted_sample_size_hypertension / (1 - loss_to_followup_rate)

# Parameters for the power analysis for diabetes patients
control_rate_diabetes = 0.50  # 50% control rate in the control arm
intervention_rate_diabetes = 0.60  # 60% control rate in the intervention arm (10% superiority)

# Calculate the effect size for proportions for diabetes patients
effect_size_diabetes = sms.proportion_effectsize(control_rate_diabetes, intervention_rate_diabetes)

# Calculate the sample size for diabetes patients
sample_size_diabetes = sms.NormalIndPower().solve_power(effect_size=effect_size_diabetes, alpha=alpha, power=power)
adjusted_sample_size_diabetes = sample_size_diabetes / (1 + (sample_size_diabetes - 1) * icc)

# Adjust the sample size for loss to follow-up
adjusted_sample_size_diabetes_with_loss = adjusted_sample_size_diabetes / (1 - loss_to_followup_rate)

# Parameters for the power analysis for HIV patients
control_rate_HIV = 0.50  # 50% control rate in the control arm
intervention_rate_HIV = 0.50  # 60% control rate in the intervention arm (10% superiority)

# Calculate the effect size for proportions for HIV patients
effect_size_HIV = sms.proportion_effectsize(control_rate_HIV, intervention_rate_HIV)

# Calculate the sample size for HIV patients
sample_size_HIV = sms.NormalIndPower().solve_power(effect_size=effect_size_HIV, alpha=alpha, power=power)
adjusted_sample_size_HIV = sample_size_HIV / (1 + (sample_size_HIV - 1) * icc)

# Adjust the sample size for loss to follow-up
adjusted_sample_size_HIV_with_loss = adjusted_sample_size_HIV / (1 - loss_to_followup_rate)

# Print the calculated values with loss to follow-up adjustment
print(f"Adjusted Sample Size per group for Hypertension Patients with Loss to Follow-up: {adjusted_sample_size_hypertension_with_loss:.2f}")
print(f"Adjusted Sample Size per group for Diabetes Patients with Loss to Follow-up: {adjusted_sample_size_diabetes_with_loss:.2f}")
print(f"Adjusted Sample Size per group for HIV Patients with Loss to Follow-up: {adjusted_sample_size_HIV_with_loss:.2f}")

# Assuming you can enroll only 8 participants per group
participants_per_group = 4

# Calculate the number of groups needed for hypertension patients with loss to follow-up adjustment
calculated_groups_hypertension_with_loss = adjusted_sample_size_hypertension_with_loss / participants_per_group

# Calculate the number of groups needed for diabetes patients with loss to follow-up adjustment
calculated_groups_diabetes_with_loss = adjusted_sample_size_diabetes_with_loss / participants_per_group

# Calculate the number of groups needed for HIV patients with loss to follow-up adjustment
calculated_groups_HIV_with_loss = adjusted_sample_size_HIV_with_loss / participants_per_group

print(f"Calculated Number of Groups for Hypertension Patients with Loss to Follow-up: {calculated_groups_hypertension_with_loss:.2f}")
print(f"Calculated Number of Groups for Diabetes Patients with Loss to Follow-up: {calculated_groups_diabetes_with_loss:.2f}")
print(f"Calculated Number of Groups for HIV Patients with Loss to Follow-up: {calculated_groups_HIV_with_loss:.2f}")

In [ ]:
(52/4 * 2) + 52/4

In [ ]:
52/4

In [ ]:
(13 * 2) + 13

In [ ]:
import statsmodels.stats.api as sms

# Parameters for the power analysis for HIV patients (non-inferiority)
control_rate_HIV = 0.50  # 50% control rate in the control arm
intervention_rate_HIV = 0.50  # Non-inferiority margin up to 10% (control rate - non-inferiority margin)
alpha = 0.05  # Significance level
power = 0.80  # Desired power level (80%)
icc = 0.02  # ICC less than 0.02
loss_to_followup_rate = 0.10  # 5% loss to follow-up rate
non_inferiority_margin = 0.10  # Non-inferiority margin

# Calculate the effect size for proportions for HIV patients (non-inferiority)
effect_size_HIV = sms.proportion_effectsize(control_rate_HIV, control_rate_HIV - non_inferiority_margin)

# Calculate the sample size for HIV patients (non-inferiority)
sample_size_HIV = sms.NormalIndPower().solve_power(effect_size=effect_size_HIV, alpha=alpha, power=power)
adjusted_sample_size_HIV = sample_size_HIV / (1 + (sample_size_HIV - 1) * icc)

# Adjust the sample size for loss to follow-up
adjusted_sample_size_HIV_with_loss = adjusted_sample_size_HIV / (1 - loss_to_followup_rate)

# Print the calculated values with loss to follow-up adjustment
print(f"Adjusted Sample Size per group for HIV Patients with Loss to Follow-up: {adjusted_sample_size_HIV_with_loss:.2f}")

# Assuming you can enroll only 8 participants per group
participants_per_group = 4

# Calculate the number of groups needed for HIV patients with loss to follow-up adjustment
calculated_groups_HIV_with_loss = adjusted_sample_size_HIV_with_loss / participants_per_group

print(f"Calculated Number of Groups for HIV Patients with Loss to Follow-up: {calculated_groups_HIV_with_loss:.2f}")

# Calculate the total number of groups for both countries
total_groups_both_countries = calculated_groups_HIV_with_loss * 2
print(f"Total Number of Groups for Both Countries: {total_groups_both_countries:.2f}")

In [ ]:
# Parameters for the power analysis for DM patients (superiority)
control_rate = 0.50  # 50% control rate in the control arm
intervention_rate = 0.60
alpha = 0.05  # Significance level
power = 0.80  # Desired power level (80%)
icc = 0.02  # ICC less than 0.02
loss_to_followup_rate = 0.10  # 10% loss to follow-up rate

# Calculate the effect size for proportions
effect_size = sms.proportion_effectsize(control_rate, intervention_rate)

# Calculate the sample size
sample_size = sms.NormalIndPower().solve_power(effect_size=effect_size, alpha=alpha, power=power)
adjusted_sample_size = sample_size / (1 + (sample_size - 1) * icc)

# Adjust the sample size for loss to follow-up
adjusted_sample_size_with_loss = adjusted_sample_size / (1 - loss_to_followup_rate)

# Print the calculated values with loss to follow-up adjustment
print(f"Adjusted Sample Size per group for HIV Patients with Loss to Follow-up: {adjusted_sample_size_with_loss:.2f}")

# Assuming you can enroll only 8 participants per group
participants_per_group = 4

# Calculate the number of groups needed with loss to follow-up adjustment
calculated_groups_with_loss = adjusted_sample_size_with_loss / participants_per_group

print(f"Calculated Number of Groups with Loss to Follow-up: {calculated_groups_with_loss:.2f}")

# Calculate the total number of groups for both countries
total_groups_both_countries = calculated_groups_with_loss * 2
print(f"Total Number of Groups for Both Countries: {total_groups_both_countries:.2f}")